# 📡 Signal Hunters R2 · v2: noisy student on the real PKS test
**Hack4Hack 2026 · Round 2 · 9-class macro-F1 on a different telescope**

**What v1 taught us:** public LB **0.9029** vs PKS-sim OOF **0.9016**. The PKS-simulated validation is calibrated, and the ×0.64 drift rescale (PKS-sim-R 0.835) is **not** present in the test data. Model A (canvas) carries the ensemble: clean 0.934, PKS-sim 0.901.

**v2 changes, all aimed at Model A and the real PKS shift:**
1. Frequency-rescale augmentation reduced to p = 0.15 in 0.85–1.15. The test has no rescale, and heavy rescaling only blurs lines and scintles.
2. The canvas gets a 7th channel `Zb` (before row-median removal). Broad, patchy scintillation brightening is no longer erased (v1 predicted only 4.8% scintillating_star vs an EM prior of 7.2%).
3. EMA weights (decay 0.998), 40 epochs. Model B is off (its blend weight was 0.05); its time goes to Model A.
4. Extra Model A seeds trained on **100% of train**, and **noisy-student** seeds trained on train + confident pseudo-labels of the **real PKS test**. The teacher is the current best ensemble, refreshed after each round.
5. Submission files are rewritten after every stage, so a crash or timeout never loses work. `submission.csv` = ensemble with pseudo-label models; `submission_alt.csv` = without them. Upload both: the leaderboard decides.

| Stage | What | Why |
|---|---|---|
| 0 | Config, seeds, data (mmap → uint8 on GPU, GBT channel-64 spike removed in the raw bytes) | reproducible; the spike must never travel with the shift/rescale augmentations |
| 1 | GPU preprocessing (missing fill, smooth bandpass flatten, panel gain, row glitch removal, robust σ) | puts GBT (σ≈11.9) and PKS (σ≈15.6) on one scale |
| 2 | GPU domain randomisation + a fixed **PKS-simulated** copy of train | the PKS-simulated OOF is our leaderboard proxy (the old notebook's proxy 0.7955 matched LB 0.7957) |
| 3 | GPU physics features (line banks as matmuls, joint ON-track search, de-dispersion, satellite, radar, scintillation, persistence) | the GBDT inputs and the CNN auxiliary vector |
| 4 | Model C: LightGBM on clean + domain-randomised feature rows | strong, cheap, complementary |
| 5 | Model A: global-time-canvas ResNet · Model B: panel-token Transformer, both with on-the-fly augmentation and on-the-fly features | sees the track and the ON/OFF contrast directly |
| 6 | AdaBN, TTA, blend weights + temperatures on PKS-sim OOF, logit biases, Saerens EM prior shift, stress tests, pseudo-labels, physics override | every choice is gated by the PKS-simulated OOF or the stress test |
| 7 | `submission.csv`, `submission_alt.csv`, `oof_and_test_probs.npz`, summary table | 2 uploads left → two genuinely different bets |

Set `CONFIG['FAST_DEBUG'] = True` for a < 10 min end-to-end smoke run on 2,000 cadences.

In [ ]:
# ===================== 0. imports, CONFIG, seeds, timer =====================
import os, sys, gc, glob, json, time, math, random, copy, warnings, subprocess, itertools
warnings.filterwarnings('ignore')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, train_test_split
import matplotlib.pyplot as plt

CONFIG = dict(
    FAST_DEBUG=False,             # True: 2,000 train / 1,000 test cadences, 2 epochs, < 10 min
    N_DEBUG_TRAIN=2000, N_DEBUG_TEST=1000,
    SEED=42,
    FOLDS=5,
    EPOCHS_A=40, EPOCHS_B=24, EPOCHS_DEBUG=2,
    BS=128, PRED_BS=512,
    LR_A=2e-3, LR_B=1e-3, WD=0.02, LABEL_SMOOTH=0.05,
    SEEDS_A=[42],                 # the 5-fold CV seed: gives the OOF that every decision is fitted on
    FULL_SEEDS=[1337, 2024],      # per seed: one full-train model, then one noisy-student model (time permitting)
    USE_MODEL_B=False,            # v1 blend weight was 0.05: time is better spent on Model A
    EMA_DECAY=0.998,
    NORM='bn',                    # 'bn' (+AdaBN on the target domain, compared on PKS-sim) or 'gn'
    BACKBONE_A='custom',          # 'custom' anisotropic ResNet, or 'timm' (resnet18 with a stride-1 stem; falls back to custom)
    SUPERPOSE_P=0.3, RESCALE_P=0.15, RESCALE_RANGE=(0.85, 1.15),
    TTA=True,
    LGB_ROUNDS=2000, LGB_DR_COPIES=1,
    USE_PSEUDO=True, PSEUDO_THR=0.90, PSEUDO_CAP=1500, PSEUDO_W=0.7,
    PHYSICS_OVERRIDE=True,
    TIME_BUDGET_MIN=330, SAFETY_MIN=20,   # Kaggle GPU sessions allow 12 h; v2 needs ~3.5 h on a T4
)
if CONFIG['FAST_DEBUG']:
    CONFIG.update(EPOCHS_A=CONFIG['EPOCHS_DEBUG'], EPOCHS_B=CONFIG['EPOCHS_DEBUG'], SEEDS_A=[42],
                  FULL_SEEDS=[1337], LGB_ROUNDS=200)
SEED = CONFIG['SEED']
T_START = time.time()
STAGE_TIMES = {}

def elapsed_min():
    return (time.time() - T_START) / 60.0

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")} | {elapsed_min():6.1f} min] {msg}', flush=True)

def seed_everything(s):
    random.seed(s); np.random.seed(s % (2 ** 32 - 1)); os.environ['PYTHONHASHSEED'] = str(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
MM_DTYPE = torch.float16 if DEVICE.type == 'cuda' else torch.float32

def amp_ctx():
    if USE_AMP:
        return torch.autocast(device_type='cuda', dtype=torch.float16)
    return torch.autocast(device_type='cpu', enabled=False)

def make_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=USE_AMP)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

# LightGBM: pip install only if Kaggle lacks it; fall back to sklearn without internet
try:
    import lightgbm as lgb
    HAVE_LGB = True
except ImportError:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True, timeout=180)
        import lightgbm as lgb
        HAVE_LGB = True
    except Exception:
        HAVE_LGB = False
        from sklearn.ensemble import HistGradientBoostingClassifier

log(f'device={DEVICE} {torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else ""} | torch {torch.__version__} '
    f'| lightgbm={"yes" if HAVE_LGB else "NO (sklearn fallback)"} | FAST_DEBUG={CONFIG["FAST_DEBUG"]}')

## 0b. Data
The data is found automatically under `/kaggle/input` (or `.` locally). Train and test are loaded with `mmap_mode='r'` and converted in batches.
The GBT channel-64 spike is replaced by the mean of channels 63/65 (plus σ/√2 noise so that the column keeps the normal noise level). This is done **in the raw bytes, before any augmentation**, so a shifted or rescaled spike can never pose as persistent RFI. The same step runs on the test set, so the preprocessing is identical.

In [ ]:
# ===================== 0b. data =====================
def find_data():
    hits = sorted(glob.glob('/kaggle/input/**/train_cadences.npy', recursive=True))
    if not hits:
        hits = sorted(glob.glob('./**/train_cadences.npy', recursive=True))
    if not hits:
        raise FileNotFoundError('train_cadences.npy not found under /kaggle/input or .')
    return os.path.dirname(hits[0])

DATA = find_data()
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
log(f'DATA={DATA}  WORK={WORK}')

CLASSES = ['eti_candidate', 'frb', 'pulsar', 'scintillating_star', 'satellite_pass',
           'radar_chirp', 'rfi_intermittent', 'rfi_persistent', 'noise']   # index = priority (0 = highest)
NC = len(CLASSES)
C2I = {c: i for i, c in enumerate(CLASSES)}
PARTNER_CLASSES = [C2I['noise'], C2I['rfi_persistent'], C2I['radar_chirp']]
ON_IDX, OFF_IDX = [0, 2, 4], [1, 3, 5]
SIG_BYTES = 11.9          # GBT robust noise sigma in byte units (measured)

X_train_mm = np.load(os.path.join(DATA, 'train_cadences.npy'), mmap_mode='r')
X_test_mm = np.load(os.path.join(DATA, 'test_cadences.npy'), mmap_mode='r')
train_meta = pd.read_csv(os.path.join(DATA, 'train_meta.csv'))
test_meta_full = pd.read_csv(os.path.join(DATA, 'test_meta.csv'))
assert len(X_train_mm) == len(train_meta) and len(X_test_mm) == len(test_meta_full)

if CONFIG['FAST_DEBUG']:
    tr_rows, _ = train_test_split(np.arange(len(train_meta)), train_size=CONFIG['N_DEBUG_TRAIN'],
                                  stratify=train_meta['label'].values, random_state=SEED)
    tr_rows = np.sort(tr_rows)
    te_rows = np.arange(CONFIG['N_DEBUG_TEST'])        # a prefix, so the ids still match test_meta.csv in order
else:
    tr_rows = np.arange(len(train_meta))
    te_rows = np.arange(len(test_meta_full))
train_df = train_meta.iloc[tr_rows].reset_index(drop=True)
test_df = test_meta_full.iloc[te_rows].reset_index(drop=True)
y = train_df['label'].map(C2I).values.astype(np.int64)
assert not np.any(pd.isna(train_df['label'].map(C2I))), 'unknown label in train'
N_TR, N_TE = len(train_df), len(test_df)

def load_raw(mm, rows, seed, chunk=2000):
    """uint8 copy with the channel-64 spike replaced by the neighbour mean + sigma/sqrt(2) noise."""
    out = np.empty((len(rows), 6, 16, 128), np.uint8)
    rng = np.random.RandomState(seed)
    for s in range(0, len(rows), chunk):
        x = np.asarray(mm[rows[s:s + chunk]]).astype(np.float32)
        miss = x == 0
        nb = 0.5 * (x[..., 63] + x[..., 65]) + rng.normal(0, SIG_BYTES / np.sqrt(2), x[..., 64].shape)
        bad = miss[..., 63] | miss[..., 65]
        x[..., 64] = np.where(bad, 0, np.clip(np.round(nb), 1, 255))
        out[s:s + len(x)] = x.astype(np.uint8)
    return out

raw_tr_np = load_raw(X_train_mm, tr_rows, SEED)
raw_te_np = load_raw(X_test_mm, te_rows, SEED + 1)
TR_RAW = torch.from_numpy(raw_tr_np).to(DEVICE)
TE_RAW = torch.from_numpy(raw_te_np).to(DEVICE)
TR_GAPS = torch.from_numpy(train_df['slew_gap_rows'].values.astype(np.int64)).clamp(3, 6).to(DEVICE)
TE_GAPS = torch.from_numpy(test_df['slew_gap_rows'].values.astype(np.int64)).clamp(3, 6).to(DEVICE)
TR_Y = torch.from_numpy(y).to(DEVICE)
TR_W = torch.ones(N_TR, device=DEVICE)

# folds: StratifiedGroupKFold grouped by target star (no same-star leakage between folds)
sgkf = StratifiedGroupKFold(n_splits=CONFIG['FOLDS'], shuffle=True, random_state=SEED)
FOLD = np.full(N_TR, -1, np.int64)
for f, (_, va) in enumerate(sgkf.split(np.zeros(N_TR), y, groups=train_df['target'].values)):
    FOLD[va] = f
assert (FOLD >= 0).all()

vc = pd.Series(y).map(dict(enumerate(CLASSES))).value_counts().reindex(CLASSES)
print(pd.DataFrame({'n': vc, 'pct': (100 * vc / N_TR).round(2)}).to_string())
print('fold sizes:', np.bincount(FOLD).tolist(), '| unique targets:', train_df['target'].nunique())
print('slew_gap_rows train', np.bincount(train_df['slew_gap_rows'], minlength=7)[3:].tolist(),
      'test', np.bincount(test_df['slew_gap_rows'], minlength=7)[3:].tolist())
log(f'loaded train {raw_tr_np.shape} test {raw_te_np.shape}')
STAGE_TIMES['load'] = elapsed_min()

## 1. Preprocessing (GPU, identical for train and test)
1. `0` is missing. Fill it with that panel's channel median and keep the mask.
2. Work in log2 power. **Bandpass**: take the per-channel median over the 6×16 rows, then a 9-channel running median across frequency. The literal per-channel median would erase every `rfi_persistent` line, because such a line sits in all 96 rows. The smoothed bandpass follows roll-off, tilt and ripple but keeps 1–3 channel lines. Then remove each panel's median (gain).
3. Subtract each row's median across channels (broadband glitches). The row profile is kept as its own channel and features. De-dispersion and scintillation use the version *before* row removal (`Zb`), so broadband physics survives.
4. Standardise per cadence by robust σ (MAD×1.4826) and clip at ±6. This is the key step for transfer: PKS (σ 15.6) and GBT (σ 11.9) land on the same scale.
5. Views: `Zlog`, `Zlin` (linear power 2^L, standardised the same way), `Ztr` (each channel's cadence median removed: shows only what *changes* between panels), `Zb`, `rowprof`, `miss`.

In [ ]:
# ===================== 1. preprocessing =====================
NAN = float('nan')

@torch.no_grad()
def preprocess(v, miss):
    """v: (B,6,16,128) float bytes (0 = missing); miss: bool mask. Returns a dict of standardised views."""
    B = v.shape[0]
    v = v.clone(); miss = miss.clone()
    bad = miss[..., 63] | miss[..., 65]                                     # channel-64 mask (both domains)
    v[..., 64] = torch.where(bad, torch.zeros_like(v[..., 64]), 0.5 * (v[..., 63] + v[..., 65]))
    miss[..., 64] = bad
    L = v / 24.0 - 4.0                                                      # log2 power
    Ln = L.masked_fill(miss, NAN)
    med_pc = torch.nanmedian(Ln, dim=2, keepdim=True).values               # (B,6,1,128)
    med_p = torch.nanmedian(Ln.reshape(B, 6, -1), dim=2).values.view(B, 6, 1, 1)
    med_pc = torch.where(torch.isnan(med_pc), med_p.expand_as(med_pc), med_pc)
    med_pc = torch.nan_to_num(med_pc, nan=0.0)
    L = torch.where(miss, med_pc.expand_as(L), L)
    bp = L.permute(0, 3, 1, 2).reshape(B, 128, 96).median(dim=-1).values   # per-channel median over 96 rows
    bp = F.pad(bp[:, None, :], (4, 4), mode='replicate')[:, 0].unfold(-1, 9, 1).median(-1).values   # smooth across freq
    L1 = L - bp[:, None, None, :]
    pm = L1.reshape(B, 6, -1).median(-1).values.view(B, 6, 1, 1)           # panel gain
    L1 = L1 - pm
    rm = L1.median(dim=-1, keepdim=True).values                             # (B,6,16,1) row profile
    Lr = L1 - rm
    Lv = Lr.masked_fill(miss, NAN).reshape(B, -1)
    c = torch.nanmedian(Lv, dim=1, keepdim=True).values
    sig = (1.4826 * torch.nanmedian((Lv - c).abs(), dim=1).values).clamp(min=1e-3).view(B, 1, 1, 1)
    c4 = c.view(B, 1, 1, 1)
    Zlog = ((Lr - c4) / sig).clamp(-6, 6).masked_fill(miss, 0.0)
    Zb = ((L1 - c4) / sig).clamp(-6, 6).masked_fill(miss, 0.0)
    rowprof = (rm[..., 0] / sig[..., 0]).clamp(-6, 6)
    P = torch.exp2(Lr)
    Pv = P.masked_fill(miss, NAN).reshape(B, -1)
    pc = torch.nanmedian(Pv, dim=1, keepdim=True).values
    ps = (1.4826 * torch.nanmedian((Pv - pc).abs(), dim=1).values).clamp(min=1e-4).view(B, 1, 1, 1)
    Zlin = ((P - pc.view(B, 1, 1, 1)) / ps).clamp(-6, 6).masked_fill(miss, 0.0)
    Ztr = Zlog - Zlog.permute(0, 3, 1, 2).reshape(B, 128, 96).median(-1).values[:, None, None, :]
    return dict(Zlog=Zlog, Zlin=Zlin, Zb=Zb, Ztr=Ztr.clamp(-6, 6), rowprof=rowprof, miss=miss.float())

## 2. Domain randomisation: make GBT look like PKS (GPU)
All of it runs on raw bytes *before* preprocessing, in this order:
**(a)** priority-respecting superposition in linear power (`p = p1 + p2 − 1`, partner ∈ {noise, rfi_persistent, radar_chirp}, label = the higher-priority one);
**(b)** geometry: frequency flip, integer shift, frequency rescale ×0.6–1.15 (covers the ×0.64 apparent-drift change if the generator is physical). The noise lost to interpolation is put back exactly (`σ·√(2w(1−w))`), and channels outside the source range are filled with noise, never mirrored lines;
**(c)** instrument: extra noise (σ ×1.0–1.45), random roll-off, ripple and tilt, panel gains, glitch rows, dropped rows, then byte quantisation.

The **PKS-simulated** copy is the same chain at fixed, strongest settings (σ ×1.3, strong low-edge roll-off, ripple, ±4.5 gains, 3× glitches, 2.6× drops). **PKS-sim-R** adds the ×0.64 rescale and is reported as a second diagnostic.

In [ ]:
# ===================== 2. augmentation (GPU) =====================
DR_TRAIN = dict(noise=(1.0, 1.45), lo_a=(0, 35), lo_w=(0, 20), hi_a=(0, 35), hi_w=(0, 20), ripple=(0, 6),
                period=(8, 64), tilt=8.0, gain=(0, 8), glitch=(0, 0.03), gamp=(5, 25), drop=(0, 0.01))
PKS_SIM = dict(noise=(1.3, 1.3), lo_a=(28, 35), lo_w=(14, 20), hi_a=(0, 12), hi_w=(0, 10), ripple=(4, 6),
               period=(10, 40), tilt=8.0, gain=(4.5, 4.5), glitch=(0.014, 0.014), gamp=(5, 25), drop=(0.0024, 0.0024))

def _U(gen, lo, hi, shape):
    return lo + (hi - lo) * torch.rand(shape, generator=gen, device=DEVICE)

def _N(gen, shape):
    return torch.randn(shape, generator=gen, device=DEVICE)

def _gather_miss(miss, I):
    return torch.gather(miss.to(torch.uint8), 3, I) > 0

def freq_resample(v, miss, s, shift, flip, gen, c0=None):
    """Resample the frequency axis: out[c] = v[c0 + (c - c0)/s - shift] (then optional flip).
    Interpolation noise loss is restored; out-of-range channels are filled with row-median + noise."""
    B = v.shape[0]
    c = torch.arange(128, device=DEVICE, dtype=torch.float32)[None]
    if c0 is None:
        c0 = torch.full((B, 1), 63.5, device=DEVICE)
    x = c0 + (c - c0) / s[:, None] - shift[:, None]
    x = torch.where(flip[:, None], 127.0 - x, x)
    inb = (x >= 0) & (x <= 127)
    xc = x.clamp(0, 127)
    i0 = xc.floor().long().clamp(max=126)
    w = xc - i0.float()
    I0 = i0[:, None, None, :].expand(B, 6, 16, 128)
    W = w[:, None, None, :]
    out = torch.gather(v, 3, I0) * (1 - W) + torch.gather(v, 3, I0 + 1) * W
    ma = _gather_miss(miss, I0) | _gather_miss(miss, I0 + 1)
    comp = torch.sqrt((2 * w * (1 - w)).clamp(min=0)) * SIG_BYTES
    out = out + comp[:, None, None, :] * _N(gen, out.shape)
    rowmed = torch.nan_to_num(torch.nanmedian(v.masked_fill(miss, NAN), dim=-1, keepdim=True).values, nan=96.0)
    fill = rowmed + SIG_BYTES * _N(gen, out.shape)
    INB = inb[:, None, None, :]
    out = torch.where(INB, out, fill)
    rowmiss = miss.all(-1, keepdim=True).expand_as(ma)
    ma = torch.where(INB, ma, rowmiss)
    return out, ma

def instrument(v, miss, P, gen, noise_mask=None):
    B = v.shape[0]
    f = _U(gen, *P['noise'], (B, 1, 1, 1))
    if noise_mask is not None:
        f = torch.where(noise_mask.view(B, 1, 1, 1), f, torch.ones_like(f))
    v = v + SIG_BYTES * torch.sqrt((f * f - 1).clamp(min=0)) * _N(gen, v.shape)
    c = torch.arange(128, device=DEVICE, dtype=torch.float32)[None]
    la, lw = _U(gen, *P['lo_a'], (B, 1)), _U(gen, *P['lo_w'], (B, 1)).clamp(min=1)
    ha, hw = _U(gen, *P['hi_a'], (B, 1)), _U(gen, *P['hi_w'], (B, 1)).clamp(min=1)
    bpc = -la * F.relu(1 - c / lw) ** 2 - ha * F.relu(1 - (127 - c) / hw) ** 2
    amp, per, ph = _U(gen, *P['ripple'], (B, 1)), _U(gen, *P['period'], (B, 1)), _U(gen, 0, 2 * math.pi, (B, 1))
    bpc = bpc + amp * torch.sin(2 * math.pi * c / per + ph) + _U(gen, -P['tilt'], P['tilt'], (B, 1)) * (c / 127 - 0.5)
    v = v + bpc[:, None, None, :]
    v = v + _U(gen, *P['gain'], (B, 1, 1, 1)) * _U(gen, -1, 1, (B, 6, 1, 1))
    gm = torch.rand((B, 6, 16, 1), generator=gen, device=DEVICE) < _U(gen, *P['glitch'], (B, 1, 1, 1))
    v = v + gm.float() * _U(gen, *P['gamp'], (B, 6, 16, 1))
    dm = torch.rand((B, 6, 16, 1), generator=gen, device=DEVICE) < _U(gen, *P['drop'], (B, 1, 1, 1))
    miss = miss | dm.expand_as(miss)
    v = v.round().clamp(1, 255).masked_fill(miss, 0.0)
    return v, miss

def superpose(v, miss, yb, pv, pm, py, sel):
    p = torch.exp2(v / 24 - 4) + torch.exp2(pv / 24 - 4) - 1
    vn = 24 * (torch.log2(p.clamp(min=2 ** (1 / 24 - 4))) + 4)
    S = sel.view(-1, 1, 1, 1)
    return torch.where(S, vn, v), torch.where(S, miss | pm, miss), torch.where(sel, torch.minimum(yb, py), yb)

@torch.no_grad()
def make_train_batch(raw, gaps, ys, ws, idx_np, pool_np, gen, aug=True):
    idx = torch.as_tensor(idx_np, device=DEVICE, dtype=torch.long)
    v = raw[idx].float(); miss = v == 0; yb = ys[idx].clone(); wb = ws[idx]; gb = gaps[idx]
    if not aug:
        return v, miss, yb, wb, gb
    B = len(idx)
    sel = torch.rand(B, generator=gen, device=DEVICE) < CONFIG['SUPERPOSE_P']
    if pool_np is not None and len(pool_np) > 0 and bool(sel.any()):
        pool = torch.as_tensor(pool_np, device=DEVICE, dtype=torch.long)
        pj = pool[torch.randint(len(pool), (B,), generator=gen, device=DEVICE)]
        pv = raw[pj].float(); pm = pv == 0
        pv, pm = freq_resample(pv, pm, torch.ones(B, device=DEVICE), _U(gen, -40, 40, (B,)).round(),
                               torch.rand(B, generator=gen, device=DEVICE) < 0.5, gen)
        v, miss, yb = superpose(v, miss, yb, pv, pm, ys[pj], sel)
    else:
        sel = torch.zeros(B, dtype=torch.bool, device=DEVICE)
    s = torch.where(torch.rand(B, generator=gen, device=DEVICE) < CONFIG['RESCALE_P'],
                    _U(gen, *CONFIG['RESCALE_RANGE'], (B,)), torch.ones(B, device=DEVICE))
    v, miss = freq_resample(v, miss, s, _U(gen, -20, 20, (B,)).round(),
                            torch.rand(B, generator=gen, device=DEVICE) < 0.5, gen, c0=_U(gen, 40, 88, (B, 1)))
    v, miss = instrument(v, miss, DR_TRAIN, gen, noise_mask=~sel)       # superposed samples are already noisier
    return v, miss, yb, wb, gb

@torch.no_grad()
def make_sim_copy(raw, P, seed, s_fixed=None, bs=1024):
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    out = torch.empty_like(raw)
    for st in range(0, len(raw), bs):
        v = raw[st:st + bs].float(); miss = v == 0; B = len(v)
        if s_fixed is not None:
            v, miss = freq_resample(v, miss, torch.full((B,), float(s_fixed), device=DEVICE),
                                    torch.zeros(B, device=DEVICE), torch.zeros(B, dtype=torch.bool, device=DEVICE), gen)
        v, miss = instrument(v, miss, P, gen)
        out[st:st + B] = v.to(torch.uint8)
    return out

@torch.no_grad()
def make_dr_copy(raw, gaps, ys, seed, bs=1024):
    """One domain-randomised copy of train (with superposition, labels updated) for the GBDT."""
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    pool = np.where(np.isin(ys.cpu().numpy(), PARTNER_CLASSES))[0]
    out = torch.empty_like(raw); yo = ys.clone()
    for st in range(0, len(raw), bs):
        idx = np.arange(st, min(len(raw), st + bs))
        v, miss, yb, _, _ = make_train_batch(raw, gaps, ys, torch.ones(len(raw), device=DEVICE), idx, pool, gen)
        out[st:st + len(idx)] = v.to(torch.uint8); yo[st:st + len(idx)] = yb
    return out, yo

PKS_RAW = make_sim_copy(TR_RAW, PKS_SIM, SEED + 101)
PKSR_RAW = make_sim_copy(TR_RAW, PKS_SIM, SEED + 202, s_fixed=0.64)
log('PKS-simulated (and PKS-sim-R, x0.64 rescale) copies of train built')

### Calibration check: does the PKS-simulated copy look like the real test set?
These are the raw statistics you measured (robust σ, gain spread, drops, glitch rows, bandpass edge, channel 64), for train, PKS-sim and test. If PKS-sim lands near test, the PKS-sim OOF is a credible leaderboard proxy.

In [ ]:
@torch.no_grad()
def domain_stats(raw, n=3000):
    v = raw[:n].float(); miss = v == 0; B = len(v)
    vn = v.masked_fill(miss, NAN)
    chmed = torch.nanmedian(vn.permute(0, 3, 1, 2).reshape(B, 128, 96), dim=-1).values
    r = vn - chmed[:, None, None, :]
    pmed = torch.nanmedian(r.reshape(B, 6, -1), dim=-1).values
    rr = r - pmed[..., None, None]
    flat = rr.reshape(B, -1)
    cc = torch.nanmedian(flat, dim=1, keepdim=True).values
    sig = 1.4826 * torch.nanmedian((flat - cc).abs(), dim=1).values
    rowm = torch.nanmedian(rr, dim=-1).values
    ok = ~torch.isnan(rowm)
    spread = (pmed.max(1).values - pmed.min(1).values)
    bp = torch.nanmedian(chmed, dim=0).values.cpu().numpy()
    return dict(sigma_bytes=float(sig.nanmedian()), gain_spread_med=float(spread.nanmedian()),
                gain_spread_p90=float(torch.nanquantile(spread, 0.9)), dropped_pct=100 * float(miss.float().mean()),
                glitch_rows_pct=100 * float(((rowm > 4.0) & ok).float().sum() / ok.float().sum()),
                bp_ch0=float(bp[0]), bp_ch64=float(bp[64]), bp_mid=float(np.median(bp))), bp

ds = {}; bps = {}
for nm, R in [('train GBT', TR_RAW), ('PKS-sim', PKS_RAW), ('PKS-sim-R', PKSR_RAW), ('test PKS', TE_RAW)]:
    ds[nm], bps[nm] = domain_stats(R)
print(pd.DataFrame(ds).round(3).to_string())
print('(train ch64 is already de-spiked in the raw bytes; the raw GBT spike was ~128 vs ~96)')
plt.figure(figsize=(10, 3))
for nm, b in bps.items():
    plt.plot(b, label=nm)
plt.title('median bandpass (bytes)'); plt.xlabel('channel'); plt.legend(); plt.tight_layout(); plt.show()
STAGE_TIMES['augment'] = elapsed_min()

## Views for the CNNs, and a sanity plot (raw → flattened → global-time canvas)
**Canvas (Model A).** The 6 panels sit at their true global rows `p·(16+gap)+r`, and the slew gaps are left as empty rows. A real ETI signal is then one straight line through ON 1, 3 and 5. Channels: Zlog, Zlin, Ztr, **Zb** (before row-median removal, keeps broad scintillation brightening), row profile, missing mask, ON/OFF indicator.
**Tokens (Model B).** For each panel: Zlog, Zlin, Ztr, the panel minus the mean of the opposite type (ON−OFF), and the missing mask.

In [ ]:
ONOFF = torch.tensor([1., -1., 1., -1., 1., -1.], device=DEVICE)
IS_ON = torch.tensor([True, False, True, False, True, False], device=DEVICE)

def canvas_rows(gaps):
    B = gaps.shape[0]
    p = torch.arange(6, device=gaps.device).view(1, 6, 1)
    r = torch.arange(16, device=gaps.device).view(1, 1, 16)
    return (p * (16 + gaps.view(B, 1, 1)) + r).reshape(B, 96)

def canvas_view(prep, gaps):
    Z = prep['Zlog']; B = Z.shape[0]
    ind = ONOFF.view(1, 6, 1, 1).expand(B, 6, 16, 128)
    ch = torch.stack([Z, prep['Zlin'], prep['Ztr'], prep['Zb'], prep['rowprof'].unsqueeze(-1).expand(B, 6, 16, 128),
                      prep['miss'], ind], 1)                                  # (B,C,6,16,128)
    C = ch.shape[1]
    X = ch.reshape(B, C, 96, 128).permute(0, 2, 1, 3)                       # (B,96,C,128)
    canvas = torch.zeros(B, 128, C, 128, device=Z.device, dtype=Z.dtype)
    canvas[torch.arange(B, device=Z.device)[:, None], canvas_rows(gaps)] = X
    return canvas.permute(0, 2, 1, 3).contiguous()                          # (B,C,128,128)

def token_view(prep):
    Z = prep['Zlog']
    other = torch.where(IS_ON.view(1, 6, 1, 1), Z[:, OFF_IDX].mean(1, keepdim=True), Z[:, ON_IDX].mean(1, keepdim=True))
    return torch.stack([Z, prep['Zlin'], prep['Ztr'], Z - other, prep['miss']], 2)   # (B,6,5,16,128)

def stack_panels(a, gap=2, fill=np.nan):
    rows = []
    for p in range(6):
        rows.append(a[p])
        if p < 5:
            rows.append(np.full((gap, a.shape[-1]), fill))
    return np.vstack(rows)

ex_idx = [int(np.where(y == c)[0][0]) for c in range(NC) if (y == c).any()]
with torch.no_grad():
    vv = TR_RAW[ex_idx].float(); pp = preprocess(vv, vv == 0); cv = canvas_view(pp, TR_GAPS[ex_idx])
fig, axes = plt.subplots(len(ex_idx), 3, figsize=(12, 2.3 * len(ex_idx)))
for r, i in enumerate(ex_idx):
    axes[r, 0].imshow(stack_panels(raw_tr_np[i].astype(float)), aspect='auto', cmap='magma', vmin=40, vmax=170)
    axes[r, 1].imshow(stack_panels(pp['Zlog'][r].cpu().numpy()), aspect='auto', cmap='magma', vmin=-3, vmax=5)
    axes[r, 2].imshow(cv[r, 0].cpu().numpy(), aspect='auto', cmap='magma', vmin=-3, vmax=5)
    axes[r, 0].set_ylabel(CLASSES[y[i]], fontsize=8)
    for a in axes[r]:
        a.set_xticks([]); a.set_yticks([])
for a, t in zip(axes[0], ['raw (6 panels, ON first)', 'flattened Zlog', 'global-time canvas']):
    a.set_title(t, fontsize=9)
plt.tight_layout(); plt.show()

## 3. Physics features (GPU, fully vectorised)
Every search is a **matrix multiply** with a precomputed, linearly interpolated "line bank". The result is an SNR (sum / √Σw²) for every (drift, centre channel).
* **Per-panel bank**: drifts −8…8 step 0.25, plus 0.05 steps inside ±1.5.
* **Joint ON-track bank**: one matrix per slew gap (3–6) and panel. It sums along `x(g) = k0 + d·(g − g_ref)` in **global time** (d ∈ ±1.5, step 0.02). It gives ON and OFF SNRs on the same track, min(ON), max(OFF), ON−OFF, the drift, and zero/non-zero-drift variants. Track residuals catch "same drift but not connected".
* **De-dispersion bank**: 45 sweep shapes (linear and 1/f²-curved, both signs, k = 0…32 rows) × 48 start rows → bursts, regularity (FFT) and the single-ON-burst FRB score.
* Satellite (steep lines, panel pattern, curvature), radar (sawtooth, periodicity, panel count), scintillation (smoothed variance, blobs, kurtosis, ON/OFF ratios), persistence (spectral correlations, shared peaks, min-over-panels line), row profile, slew gap.

In [ ]:
D1 = np.unique(np.round(np.concatenate([np.arange(-8, 8.001, 0.25), np.arange(-1.5, 1.5001, 0.05)]), 3)).astype(np.float32)
D2 = np.round(np.arange(-1.5, 1.5001, 0.02), 3).astype(np.float32)
ND1, ND2 = len(D1), len(D2)

def line_matrix(trow, drifts, min_rows):
    D = len(drifts); k = np.arange(128)
    W = np.zeros((16 * 128, D * 128), np.float32); n = np.zeros(D * 128, np.float32)
    cols = np.arange(D)[:, None] * 128 + k[None, :]
    for r in range(16):
        x = k[None, :] + drifts[:, None] * trow[r]
        inb = (x >= -0.5) & (x <= 127.5)
        x0 = np.floor(x).astype(np.int64); w1 = (x - x0).astype(np.float32)
        for cc, ww in ((x0, 1 - w1), (x0 + 1, w1)):
            ok = inb & (cc >= 0) & (cc <= 127) & (ww > 1e-6)
            np.add.at(W, (r * 128 + cc[ok], cols[ok]), ww[ok])
        n += inb.ravel()
    norm = np.sqrt((W ** 2).sum(0))
    return W, np.maximum(norm, 1e-6), n >= min_rows

class Banks:
    pass

BK = Banks()
_W, _n, _v = line_matrix(np.arange(16) - 7.5, D1, 8)
BK.pb_W = torch.from_numpy(_W).to(DEVICE, MM_DTYPE); BK.pb_norm = torch.from_numpy(_n).to(DEVICE)
BK.pb_valid = torch.from_numpy(_v).to(DEVICE)
BK.D1 = torch.from_numpy(D1).to(DEVICE); BK.D1_abs = BK.D1.abs()
BK.D2 = torch.from_numpy(D2).to(DEVICE); BK.D2_abs = BK.D2.abs()
BK.gt = {}
for G in (3, 4, 5, 6):
    g_ref = 2 * (16 + G) + 7.5
    Ws, ns, vs = [], [], []
    for p in range(6):
        _W, _n, _v = line_matrix(p * (16 + G) + np.arange(16) - g_ref, D2, 8 if p % 2 == 0 else 1)
        Ws.append(torch.from_numpy(_W).to(DEVICE, MM_DTYPE)); ns.append(torch.from_numpy(_n)); vs.append(torch.from_numpy(_v))
    BK.gt[G] = (torch.stack(Ws), torch.stack(ns).to(DEVICE), torch.stack(vs).to(DEVICE))
    del Ws
del _W; gc.collect()

# de-dispersion bank
DD_K = [1, 2, 3, 4, 6, 8, 11, 14, 18, 24, 32]
_cn = np.arange(128) / 127.0
_g_lin = 1 - _cn
_g_cur = ((_cn + 1) ** -2 - 0.25) / 0.75
_cfgs, _absk = [np.zeros(128)], [0]
for K in DD_K:
    for shape in (_g_lin, _g_cur):
        for sgn in (1, -1):
            _cfgs.append(K * (shape if sgn > 0 else shape[::-1])); _absk.append(K)
T0 = np.arange(-32, 16); NT = len(T0); NCFG = len(_cfgs)
_W = np.zeros((2048, NCFG * NT), np.float32); _n = np.zeros(NCFG * NT, np.float32)
_C = np.broadcast_to(np.arange(128)[None, :], (NT, 128))
_tcol = np.broadcast_to(np.arange(NT)[:, None], (NT, 128))
for j, delay in enumerate(_cfgs):
    rr = T0[:, None] + delay[None, :]
    inb = (rr >= -0.5) & (rr <= 15.5)
    r0 = np.floor(rr).astype(np.int64); w1 = (rr - r0).astype(np.float32)
    for ri, ww in ((r0, 1 - w1), (r0 + 1, w1)):
        ok = inb & (ri >= 0) & (ri <= 15) & (ww > 1e-6)
        np.add.at(_W, (ri[ok] * 128 + _C[ok], j * NT + _tcol[ok]), ww[ok])
    _n[j * NT:(j + 1) * NT] = inb.sum(1)
BK.dd_W = torch.from_numpy(_W).to(DEVICE, MM_DTYPE)
BK.dd_norm = torch.from_numpy(np.maximum(np.sqrt((_W ** 2).sum(0)), 1e-6)).to(DEVICE)
BK.dd_valid = torch.from_numpy(_n >= 24).to(DEVICE)
BK.dd_absk = torch.tensor(_absk, device=DEVICE, dtype=torch.float32)
del _W
_t = (np.arange(16) - 7.5) / 7.5
BK.pinv2 = torch.linalg.pinv(torch.tensor(np.stack([np.ones(16), _t, _t ** 2], 1), dtype=torch.float32)).to(DEVICE)
BK.pinv1 = torch.linalg.pinv(torch.tensor(np.stack([np.ones(16), _t], 1), dtype=torch.float32)).to(DEVICE)
BK.X2 = torch.tensor(np.stack([np.ones(16), _t, _t ** 2], 1), dtype=torch.float32, device=DEVICE)
BK.X1 = torch.tensor(np.stack([np.ones(16), _t], 1), dtype=torch.float32, device=DEVICE)
log(f'line banks built: panel {ND1} drifts, global {ND2} drifts x 4 gaps, de-dispersion {NCFG} shapes x {NT} starts')

In [ ]:
def _best(Sm):
    """Sm (B,P,D,128) -> value, drift index, channel (B,P)."""
    B, P = Sm.shape[:2]
    v, i = Sm.reshape(B, P, -1).max(-1)
    return v, i // 128, i % 128

@torch.no_grad()
def featurize(prep, gaps):
    Z, Zb, rp = prep['Zlog'], prep['Zb'], prep['rowprof']
    B = Z.shape[0]; dev = Z.device
    Fd = {}
    kk = torch.arange(128, device=dev).view(1, 1, 128)
    ar = torch.arange(B, device=dev)
    # ---------------- per-panel line bank ----------------
    S = (Z.reshape(B * 6, 2048).to(MM_DTYPE) @ BK.pb_W).float() / BK.pb_norm
    S = S.masked_fill(~BK.pb_valid, -10.0).view(B, 6, ND1, 128)
    a1 = BK.D1_abs.view(1, 1, -1, 1)
    pb_v, pb_di, pb_k = _best(S)
    Spos = S.max(2).values
    pb_2nd = Spos.masked_fill((kk - pb_k.unsqueeze(-1)).abs() < 5, -10.0).max(-1).values
    Sst = S.masked_fill(a1 <= 3, -10.0)
    st_v, st_di, st_k = _best(Sst)
    st_2nd = Sst.max(2).values.masked_fill((kk - st_k.unsqueeze(-1)).abs() < 10, -10.0).max(-1).values
    del Sst
    Smod = S.masked_fill(a1 > 1.6, -10.0)
    md_v, md_di, md_k = _best(Smod)
    Smod_pos = Smod.max(2).values
    del Smod
    zd_v = S.masked_fill(a1 >= 0.1, -10.0).reshape(B, 6, -1).max(-1).values
    del S
    for p in range(6):
        Fd[f'pb_best_p{p}'] = pb_v[:, p]; Fd[f'pb_absd_p{p}'] = BK.D1_abs[pb_di[:, p]]
        Fd[f'pb_2nd_p{p}'] = pb_2nd[:, p]; Fd[f'pb_steep_p{p}'] = st_v[:, p]
        Fd[f'pb_steep2_p{p}'] = st_2nd[:, p]; Fd[f'pb_zero_p{p}'] = zd_v[:, p]
    Fd['pb_on_min'] = pb_v[:, ON_IDX].min(1).values; Fd['pb_on_max'] = pb_v[:, ON_IDX].max(1).values
    Fd['pb_off_min'] = pb_v[:, OFF_IDX].min(1).values; Fd['pb_off_max'] = pb_v[:, OFF_IDX].max(1).values
    Fd['pb_onmin_offmax'] = Fd['pb_on_min'] - Fd['pb_off_max']
    Fd['zero_on_min'] = zd_v[:, ON_IDX].min(1).values; Fd['zero_off_max'] = zd_v[:, OFF_IDX].max(1).values
    Fd['zero_onmin_offmax'] = Fd['zero_on_min'] - Fd['zero_off_max']
    # ---------------- satellite ----------------
    hit = (st_v > 6).float()
    nsat = hit.sum(1); consec = (hit[:, :-1] * hit[:, 1:]).sum(1)
    Fd['sat_npanels'] = nsat
    Fd['sat_consec'] = ((nsat == 1) | ((nsat == 2) & (consec == 1))).float()
    Fd['sat_max'] = st_v.max(1).values
    ps = st_v.argmax(1)
    Fd['sat_absd'] = BK.D1_abs[st_di[ar, ps]]
    Zp = Z[ar, ps]                                                       # (B,16,128)
    am = Zp.argmax(-1).float(); amx = Zp.max(-1).values
    c2 = am @ BK.pinv2.T; c1 = am @ BK.pinv1.T
    res2 = (am - c2 @ BK.X2.T).std(1); res1 = (am - c1 @ BK.X1.T).std(1)
    Fd['sat_curv'] = c2[:, 2].abs(); Fd['sat_res1'] = res1; Fd['sat_res2'] = res2; Fd['sat_res_gain'] = res1 - res2
    Fd['sat_rowmax_mean'] = amx.mean(1)
    # ---------------- radar ----------------
    am_all = Z.argmax(-1).float(); mx_all = Z.max(-1).values             # (B,6,16)
    dl = am_all[..., 1:] - am_all[..., :-1]
    jumps = (dl.abs() > 40).float().sum(-1)
    med_d = dl.median(-1).values
    consist = ((torch.sign(dl) == torch.sign(med_d).unsqueeze(-1)) & (dl.abs() >= 2)).float().mean(-1)
    saw = consist * (jumps >= 1).float()
    a0 = am_all - am_all.mean(-1, keepdim=True); den = (a0 * a0).sum(-1).clamp(min=1e-6)
    ac = torch.stack([(a0[..., :-L] * a0[..., L:]).sum(-1) / den for L in range(2, 9)], -1).max(-1).values
    Fd['radar_saw_on'] = saw[:, ON_IDX].mean(1); Fd['radar_saw_off'] = saw[:, OFF_IDX].mean(1)
    Fd['radar_consist'] = consist.mean(1); Fd['radar_jumps'] = jumps.sum(1)
    Fd['radar_ac'] = ac.mean(1); Fd['radar_ac_min'] = ac.min(1).values
    Fd['radar_np'] = ((st_v > 5) & (st_2nd > 4)).float().sum(1)
    Fd['radar_nsteep5'] = (st_v > 5).float().sum(1)
    Fd['rowmax_on'] = mx_all[:, ON_IDX].mean((1, 2)); Fd['rowmax_off'] = mx_all[:, OFF_IDX].mean((1, 2))
    Fd['rowmax_min_panel'] = mx_all.mean(-1).min(1).values
    # ---------------- joint ON-track search in global time ----------------
    gt = torch.empty(B, 6, ND2, 128, device=dev)
    fillv = torch.tensor([-10., 0., -10., 0., -10., 0.], device=dev).view(6, 1, 1)
    for G in (3, 4, 5, 6):
        sel = (gaps == G).nonzero(as_tuple=True)[0]
        if sel.numel() == 0:
            continue
        Wg, ng, vg = BK.gt[G]
        Zs = Z[sel].reshape(-1, 6, 2048).transpose(0, 1).to(MM_DTYPE)
        Sg = torch.bmm(Zs, Wg).float() / ng.unsqueeze(1)
        Sg = torch.where(vg.unsqueeze(1), Sg, fillv)
        gt[sel] = Sg.transpose(0, 1).reshape(-1, 6, ND2, 128)
    on_t, off_t = gt[:, ON_IDX], gt[:, OFF_IDX]
    s_on = on_t.sum(1) / math.sqrt(3); s_off = off_t.sum(1) / math.sqrt(3)
    mn_on = on_t.min(1).values; mx_off = off_t.max(1).values
    del gt, on_t, off_t
    con = mn_on - mx_off
    a2 = BK.D2_abs.view(1, -1, 1)
    nzm, zm = a2 >= 0.1, a2 < 0.1

    def at_best(score, mask=None):
        sc = score if mask is None else score.masked_fill(~mask, -1e4)
        v, i = sc.reshape(B, -1).max(1)
        return v, i // 128, i % 128

    for name, score in (('sum', s_on), ('min', mn_on), ('con', con)):
        v, di, k = at_best(score)
        Fd[f'gt_{name}_best'] = v; Fd[f'gt_{name}_absd'] = BK.D2_abs[di]
        Fd[f'gt_{name}_onmin'] = mn_on[ar, di, k]; Fd[f'gt_{name}_offmax'] = mx_off[ar, di, k]
        Fd[f'gt_{name}_onsum'] = s_on[ar, di, k]; Fd[f'gt_{name}_offsum'] = s_off[ar, di, k]
        Fd[f'gt_{name}_zero'] = (BK.D2_abs[di] < 0.1).float()
        v, di, k = at_best(score, nzm)
        Fd[f'gt_{name}_nz'] = v; Fd[f'gt_{name}_nz_offmax'] = mx_off[ar, di, k]; Fd[f'gt_{name}_nz_absd'] = BK.D2_abs[di]
        Fd[f'gt_{name}_z'] = at_best(score, zm)[0]
    Fd['gt_sum_onoff'] = Fd['gt_sum_onsum'] - Fd['gt_sum_offsum']
    Fd['gt_con_nz_minus_z'] = Fd['gt_con_nz'] - Fd['gt_con_z']
    # track residuals (independent ON fits vs the joint track) and the presence pattern
    v, di, k0 = at_best(con)
    dstar = BK.D2[di]
    Gf = gaps.float(); g_ref = 2 * (16 + Gf) + 7.5
    gc = torch.stack([p * (16 + Gf) + 7.5 for p in range(6)], 1)
    pred = k0.float().unsqueeze(1) + dstar.unsqueeze(1) * (gc - g_ref.unsqueeze(1))   # (B,6)
    kp = md_k.float()
    res = (kp - pred).abs()[:, ON_IDX]
    Fd['trk_res_max'] = res.max(1).values; Fd['trk_res_mean'] = res.mean(1)
    k_on = kp[:, ON_IDX]
    Fd['trk_lin_res'] = (k_on[:, 0] - 2 * k_on[:, 1] + k_on[:, 2]).abs() / math.sqrt(6)
    fit_d = (k_on[:, 2] - k_on[:, 0]) / (4 * (16 + Gf))
    pd_on = BK.D1[md_di[:, ON_IDX]]
    Fd['trk_drift_mismatch'] = (fit_d - pd_on.mean(1)).abs()
    Fd['trk_drift_std'] = pd_on.std(1)
    Fd['trk_abs_mean_drift'] = pd_on.mean(1).abs()
    Fd['trk_ind_onmin'] = md_v[:, ON_IDX].min(1).values; Fd['trk_ind_offmax'] = md_v[:, OFF_IDX].max(1).values
    win = (kk - pred.unsqueeze(-1)).abs() <= 5
    pres = Smod_pos.masked_fill(~win, -10.0).max(-1).values
    pres = torch.where(pres <= -9.99, torch.zeros_like(pres), pres)
    pw = (2 ** torch.arange(6, device=dev)).float()
    for p in range(6):
        Fd[f'pres_p{p}'] = pres[:, p]
    for thr in (4, 5, 6):
        hb = (pres > thr).float()
        Fd[f'pres_bits{thr}'] = (hb * pw).sum(1)
        Fd[f'pres_non{thr}'] = hb[:, ON_IDX].sum(1); Fd[f'pres_noff{thr}'] = hb[:, OFF_IDX].sum(1)
    # ---------------- persistence ----------------
    spec = Z.mean(2) * 4.0
    spec_s = F.max_pool1d(spec.reshape(B * 6, 1, 128), 3, 1, 1).view(B, 6, 128)
    sn = spec - spec.mean(-1, keepdim=True); sn = sn / sn.norm(dim=-1, keepdim=True).clamp(min=1e-6)
    cc = sn @ sn.transpose(1, 2)
    onon = torch.stack([cc[:, i, j] for i, j in ((0, 2), (0, 4), (2, 4))], 1)
    offoff = torch.stack([cc[:, i, j] for i, j in ((1, 3), (1, 5), (3, 5))], 1)
    onoff = torch.stack([cc[:, i, j] for i in ON_IDX for j in OFF_IDX], 1)
    Fd['cc_onon'] = onon.mean(1); Fd['cc_offoff'] = offoff.mean(1); Fd['cc_onoff'] = onoff.mean(1)
    Fd['cc_min'] = torch.cat([onon, offoff, onoff], 1).min(1).values
    Fd['cc_onon_minus_onoff'] = Fd['cc_onon'] - Fd['cc_onoff']
    pk = spec_s.argmax(-1); gpk = spec.mean(1).argmax(-1)
    Fd['pk_agree'] = ((pk - gpk.unsqueeze(1)).abs() <= 3).float().mean(1)
    srt = spec_s.sort(1).values
    Fd['persist_min6'] = srt[:, 0].max(-1).values; Fd['persist_5of6'] = srt[:, 1].max(-1).values
    Fd['persist_4of6'] = srt[:, 2].max(-1).values
    Fd['spec_on_only'] = (spec_s[:, ON_IDX].min(1).values - spec_s[:, OFF_IDX].max(1).values).max(-1).values
    Fd['spec_off_only'] = (spec_s[:, OFF_IDX].min(1).values - spec_s[:, ON_IDX].max(1).values).max(-1).values
    for p in range(6):
        Fd[f'spec_max_p{p}'] = spec_s[:, p].max(-1).values
    # ---------------- scintillation / texture (on Zb: before row removal) ----------------
    zb = Zb.reshape(B * 6, 1, 16, 128)
    a35 = F.avg_pool2d(zb, (3, 5), 1)
    v35 = (a35.var(dim=(1, 2, 3)) * 15).view(B, 6)
    v515 = (F.avg_pool2d(zb, (5, 15), 1).var(dim=(1, 2, 3)) * 75).view(B, 6)
    blob = (a35 * math.sqrt(15) > 3).float().mean((1, 2, 3)).view(B, 6)
    blk = (F.avg_pool2d(zb, (4, 16)).std(dim=(1, 2, 3)) * 8).view(B, 6)
    zc = zb - zb.mean((2, 3), keepdim=True)
    kurt = ((zc ** 4).mean((1, 2, 3)) / ((zc ** 2).mean((1, 2, 3)) ** 2).clamp(min=1e-6) - 3).view(B, 6)
    for nm, t, positive in (('v35', v35, True), ('v515', v515, True), ('blob', blob, False), ('blk', blk, True), ('kurt', kurt, False)):
        on, off = t[:, ON_IDX], t[:, OFF_IDX]
        Fd[f'scint_{nm}_on'] = on.mean(1); Fd[f'scint_{nm}_off'] = off.mean(1)
        Fd[f'scint_{nm}_onmin'] = on.min(1).values; Fd[f'scint_{nm}_offmax'] = off.max(1).values
        if positive:
            Fd[f'scint_{nm}_lr'] = torch.log(on.mean(1).clamp(min=1e-3) / off.mean(1).clamp(min=1e-3))
        else:
            Fd[f'scint_{nm}_diff'] = on.mean(1) - off.mean(1)
    for p in range(6):
        Fd[f'scint_v515_p{p}'] = v515[:, p]
    # ---------------- de-dispersion (on Zb) ----------------
    Sd = (Zb.reshape(B * 6, 2048).to(MM_DTYPE) @ BK.dd_W).float() / BK.dd_norm
    Sd = Sd.masked_fill(~BK.dd_valid, -10.0).view(B, 6, NCFG, NT)
    k0s = Sd[:, :, 0].max(-1).values
    dmax, di = Sd[:, :, 1:].reshape(B, 6, -1).max(-1)
    cfg = di // NT + 1
    absk = BK.dd_absk[cfg]
    series = Sd[ar[:, None], torch.arange(6, device=dev)[None, :], cfg]          # (B,6,NT)
    mid = series[..., 1:-1]
    lm = (mid > 5) & (mid >= series[..., :-2]) & (mid >= series[..., 2:])
    nbur = lm.float().sum(-1)
    sin_ = series[..., 32:48]
    sin_ = sin_ - sin_.mean(-1, keepdim=True)
    pwr = torch.fft.rfft(sin_, dim=-1).abs() ** 2
    per = pwr[..., 1:].max(-1).values / pwr[..., 1:].sum(-1).clamp(min=1e-6)
    for p in range(6):
        Fd[f'dd_max_p{p}'] = dmax[:, p]; Fd[f'dd_k0_p{p}'] = k0s[:, p]
        Fd[f'dd_absk_p{p}'] = absk[:, p]; Fd[f'dd_nb_p{p}'] = nbur[:, p]
    don = dmax[:, ON_IDX].sort(1, descending=True).values
    Fd['dd_on_max'] = don[:, 0]; Fd['dd_on_2nd'] = don[:, 1]; Fd['dd_on_min'] = don[:, 2]
    Fd['dd_off_max'] = dmax[:, OFF_IDX].max(1).values
    Fd['dd_n_on5'] = (dmax[:, ON_IDX] > 5).float().sum(1); Fd['dd_n_off5'] = (dmax[:, OFF_IDX] > 5).float().sum(1)
    Fd['frb_score'] = don[:, 0] - don[:, 1]
    Fd['dd_nb_on'] = nbur[:, ON_IDX].sum(1); Fd['dd_nb_off'] = nbur[:, OFF_IDX].sum(1)
    Fd['dd_per_on'] = per[:, ON_IDX].mean(1); Fd['dd_per_off'] = per[:, OFF_IDX].mean(1)
    Fd['dd_k_std_on'] = absk[:, ON_IDX].std(1)
    Fd['dd_k0_on'] = k0s[:, ON_IDX].max(1).values; Fd['dd_k0_off'] = k0s[:, OFF_IDX].max(1).values
    Fd['dd_on_vs_k0'] = Fd['dd_on_max'] - Fd['dd_k0_on']
    # ---------------- row profile ----------------
    for p in range(6):
        Fd[f'rp_max_p{p}'] = rp[:, p].max(-1).values
    Fd['rp_n3_on'] = (rp[:, ON_IDX] > 3).float().sum((1, 2)); Fd['rp_n3_off'] = (rp[:, OFF_IDX] > 3).float().sum((1, 2))
    Fd['slew_gap_rows'] = gaps.float()
    names = list(Fd.keys())
    X = torch.nan_to_num(torch.stack([Fd[n].float() for n in names], 1), nan=0.0, posinf=50.0, neginf=-50.0)
    return X, names

@torch.no_grad()
def featurize_raw(raw, gaps, bs=None):
    bs = bs or CONFIG['PRED_BS']; out = []
    for st in range(0, len(raw), bs):
        v = raw[st:st + bs].float()
        X, names = featurize(preprocess(v, v == 0), gaps[st:st + bs])
        out.append(X.cpu().numpy())
    return np.concatenate(out), names

t0 = time.time()
F_tr, FEAT_NAMES = featurize_raw(TR_RAW, TR_GAPS)
log(f'features train {F_tr.shape} in {time.time() - t0:.1f}s ({len(FEAT_NAMES)} features)')
F_te, _ = featurize_raw(TE_RAW, TE_GAPS)
F_pks, _ = featurize_raw(PKS_RAW, TR_GAPS)
F_pksR, _ = featurize_raw(PKSR_RAW, TR_GAPS)
F_dr, y_dr = [], []
for kcopy in range(CONFIG['LGB_DR_COPIES']):
    DR_RAW, DR_Y = make_dr_copy(TR_RAW, TR_GAPS, TR_Y, SEED + 303 + kcopy)
    F_dr.append(featurize_raw(DR_RAW, TR_GAPS)[0]); y_dr.append(DR_Y.cpu().numpy())
    del DR_RAW
torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
NF = len(FEAT_NAMES)
FN_MED = np.median(F_tr, 0)
_q75, _q25 = np.percentile(F_tr, [75, 25], 0)
FN_SCALE = np.where(_q75 - _q25 > 1e-6, _q75 - _q25, F_tr.std(0) + 1e-6)
FN_MED_T = torch.tensor(FN_MED, dtype=torch.float32, device=DEVICE)
FN_SCALE_T = torch.tensor(FN_SCALE, dtype=torch.float32, device=DEVICE)

def fnorm(f):
    return ((f - FN_MED_T) / FN_SCALE_T).clamp(-5, 5)

Ftr_df = pd.DataFrame(F_tr, columns=FEAT_NAMES)
KEY = ['gt_con_best', 'gt_con_nz', 'gt_con_absd', 'gt_min_best', 'trk_res_max', 'trk_lin_res', 'persist_min6',
       'pres_non5', 'pres_noff5', 'dd_on_max', 'frb_score', 'dd_nb_on', 'sat_max', 'sat_consec', 'radar_np',
       'radar_saw_on', 'scint_v515_lr', 'scint_blob_diff', 'cc_onon_minus_onoff']
print('median of key features per class (physics sanity check):')
print(Ftr_df[KEY].assign(label=[CLASSES[i] for i in y]).groupby('label').median().reindex(CLASSES).round(2).to_string())
print('\ndomain shift check, medians: train / PKS-sim / test')
print(pd.DataFrame({'train': np.median(F_tr, 0), 'PKS-sim': np.median(F_pks, 0), 'test': np.median(F_te, 0)},
                   index=FEAT_NAMES).loc[KEY].round(3).T.to_string())
log('features done')
STAGE_TIMES['features'] = elapsed_min()

## Metrics and stress-test helpers
Three numbers for every model: **clean OOF**, **PKS-sim OOF** (the decision number) and the **prior-shift stress test**. The stress test resamples the PKS-sim OOF to a uniform class mix and to 3 Dirichlet mixes, because the test label mix is different and unknown. PKS-sim-R (with the ×0.64 frequency rescale) is printed as an extra diagnostic.

In [ ]:
def per_class_f1(yt, yp):
    cm = np.bincount(yt * NC + yp, minlength=NC * NC).reshape(NC, NC).astype(float)
    tp = np.diag(cm); den = cm.sum(0) + cm.sum(1)
    return np.where(den > 0, 2 * tp / np.maximum(den, 1e-9), 0.0)

def mf1(yt, yp):
    return float(per_class_f1(yt, yp).mean())

def print_f1(title, yt, yp):
    f = per_class_f1(yt, yp)
    print(f'--- {title}: macro-F1 = {f.mean():.4f}')
    for c in range(NC):
        print(f'  {CLASSES[c]:20s} {f[c]:.3f} ' + '█' * int(round(f[c] * 40)))

def print_cm(title, yt, yp):
    cm = np.bincount(yt * NC + yp, minlength=NC * NC).reshape(NC, NC)
    short = ['eti', 'frb', 'psr', 'scint', 'sat', 'radar', 'rfi_int', 'rfi_per', 'noise']
    print(f'--- confusion matrix ({title}); rows = true, cols = predicted')
    print(pd.DataFrame(cm, index=short, columns=short).to_string())
    for a, b in [(0, 6), (1, 2), (4, 5), (3, 8)]:
        print(f'  {CLASSES[a]} -> {CLASSES[b]}: {cm[a, b]:4d} | {CLASSES[b]} -> {CLASSES[a]}: {cm[b, a]:4d}')

def _mixes():
    rs = np.random.RandomState(2026)
    mixes = [('uniform', np.full(NC, 1.0 / NC))]
    for k in range(3):
        m = np.clip(rs.dirichlet(np.ones(NC)), 0.02, None); mixes.append((f'dirichlet{k}', m / m.sum()))
    return mixes

MIXES = _mixes()

def resample_idx(yt, mix, n=10000, seed=0):
    rs = np.random.RandomState(seed); out = []
    for c in range(NC):
        pool = np.where(yt == c)[0]
        k = int(round(n * mix[c]))
        if len(pool) and k:
            out.append(rs.choice(pool, k, replace=True))
    return np.concatenate(out)

STRESS_IDX = [resample_idx(y, m, seed=i) for i, (_, m) in enumerate(MIXES)]

def em_prior(p, src_prior, iters=300, tol=1e-7):
    """Saerens et al. (2002): EM estimate of the target prior and the re-weighted posteriors."""
    p = np.clip(p, 1e-9, 1); pi = src_prior.copy()
    q = p
    for _ in range(iters):
        q = p * (pi / src_prior); q = q / q.sum(1, keepdims=True)
        new = q.mean(0)
        if np.abs(new - pi).max() < tol:
            pi = new; break
        pi = new
    return q, pi

def decide(p, bias=None, use_em=False, src_prior=None):
    q = em_prior(p, src_prior)[0] if use_em else p
    s = np.log(np.clip(q, 1e-9, 1))
    if bias is not None:
        s = s + bias
    return s.argmax(1), s

def stress_f1(p_oof, bias=None, use_em=False, src_prior=None):
    vals = [mf1(y[ix], decide(p_oof[ix], bias, use_em, src_prior)[0]) for ix in STRESS_IDX]
    return float(np.mean(vals)), vals

def three_numbers(name, d):
    """d: dict with clean / pks / pksR OOF probabilities."""
    out = dict(model=name, clean=mf1(y, d['clean'].argmax(1)), pks_sim=mf1(y, d['pks'].argmax(1)),
               pks_sim_R=mf1(y, d['pksR'].argmax(1)), stress=stress_f1(d['pks'])[0])
    log(f"{name:10s} clean {out['clean']:.4f} | PKS-sim {out['pks_sim']:.4f} | PKS-sim-R {out['pks_sim_R']:.4f} | stress {out['stress']:.4f}")
    return out

RESULTS = {}      # name -> dict(clean, pks, pksR, test) probabilities (+ variants)
RUNTIME = {}

## 4. Model C: LightGBM on the physics features
Each fold trains on the clean rows **plus** one domain-randomised copy of the same training cadences (superposition, rescale, PKS-like instrument). Classes are balanced with weights. Early stopping uses the clean validation fold. OOF is reported on the clean, PKS-sim and PKS-sim-R versions of the same held-out cadences.

In [ ]:
t_c = time.time()
cw_bal = (len(y) / (NC * np.maximum(np.bincount(y, minlength=NC), 1))).astype(np.float32)
LGB_PARAMS = dict(objective='multiclass', num_class=NC, learning_rate=0.04, num_leaves=48, min_data_in_leaf=25,
                  feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1, lambda_l2=2.0, verbose=-1,
                  num_threads=os.cpu_count() or 4, seed=SEED, deterministic=True, force_row_wise=True)
oofC = {k: np.zeros((N_TR, NC)) for k in ('clean', 'pks', 'pksR')}
testC = np.zeros((N_TE, NC))
gain_imp = np.zeros(NF)
for f in range(CONFIG['FOLDS']):
    tr, va = np.where(FOLD != f)[0], np.where(FOLD == f)[0]
    Xtr = np.vstack([F_tr[tr]] + [Fd_[tr] for Fd_ in F_dr])
    ytr = np.concatenate([y[tr]] + [yd[tr] for yd in y_dr])
    wtr = cw_bal[ytr]
    if HAVE_LGB:
        dtr = lgb.Dataset(Xtr, ytr, weight=wtr, free_raw_data=True)
        dva = lgb.Dataset(F_tr[va], y[va], reference=dtr)
        m = lgb.train(LGB_PARAMS, dtr, CONFIG['LGB_ROUNDS'], valid_sets=[dva],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        pr = lambda X: m.predict(X, num_iteration=m.best_iteration)
        gain_imp += m.feature_importance('gain'); bi = m.best_iteration
    else:
        m = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.06, class_weight='balanced', random_state=SEED)
        m.fit(Xtr, ytr); pr = m.predict_proba; bi = 400
    oofC['clean'][va] = pr(F_tr[va]); oofC['pks'][va] = pr(F_pks[va]); oofC['pksR'][va] = pr(F_pksR[va])
    testC += pr(F_te) / CONFIG['FOLDS']
    log(f'LGB fold {f}: iters {bi} | clean {mf1(y[va], oofC["clean"][va].argmax(1)):.4f} '
        f'| PKS-sim {mf1(y[va], oofC["pks"][va].argmax(1)):.4f}')
RESULTS['C'] = dict(clean=oofC['clean'], pks=oofC['pks'], pksR=oofC['pksR'], test=testC)
RUNTIME['C'] = (time.time() - t_c) / 60
if HAVE_LGB:
    print('top-25 features by gain:')
    print(pd.Series(gain_imp, index=FEAT_NAMES).sort_values(ascending=False).head(25).round(0).to_string())
print_f1('LightGBM clean OOF', y, oofC['clean'].argmax(1))
print_f1('LightGBM PKS-sim OOF', y, oofC['pks'].argmax(1))
SUMMARY_ROWS = [three_numbers('C (LGB)', RESULTS['C'])]
STAGE_TIMES['lgb'] = elapsed_min()

## 5. Model A (and optional Model B)
**Model A: global-time canvas ResNet.** A custom ResNet with anisotropic 3×5 / 3×7 kernels and a stride-1 stem, so 1–3-channel lines survive the first layer. Input is 7 canvas channels; avg+max pooling; the standardised physics vector is concatenated before the head.
Every training step sees a freshly augmented batch. Augmentation, preprocessing **and the physics features** are recomputed on the GPU, so the auxiliary vector always matches the image. **EMA** of the weights is used for prediction.
Model B (panel tokens) is kept in the code behind `USE_MODEL_B`.

In [ ]:
def Norm(c):
    return nn.GroupNorm(8, c) if CONFIG['NORM'] == 'gn' else nn.BatchNorm2d(c)

class Block(nn.Module):
    def __init__(self, cin, cout, stride, k=(3, 5)):
        super().__init__()
        pad = (k[0] // 2, k[1] // 2)
        self.c1 = nn.Conv2d(cin, cout, k, stride, pad, bias=False); self.b1 = Norm(cout)
        self.c2 = nn.Conv2d(cout, cout, k, 1, pad, bias=False); self.b2 = Norm(cout)
        self.sc = nn.Identity() if (stride in (1, (1, 1)) and cin == cout) else \
            nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), Norm(cout))
    def forward(self, x):
        return F.relu(self.b2(self.c2(F.relu(self.b1(self.c1(x))))) + self.sc(x))

class CanvasNet(nn.Module):
    def __init__(self, nf, cin=7):
        super().__init__()
        self.body, dim = None, 512
        if CONFIG['BACKBONE_A'] == 'timm':
            try:
                import timm
                tb = timm.create_model('resnet18', pretrained=False, in_chans=cin, num_classes=0, global_pool='')
                tb.conv1.stride = (1, 1); tb.maxpool = nn.Identity()
                self.body, dim = tb, 2 * tb.num_features
            except Exception as e:
                print('timm unavailable, using the custom ResNet:', e)
        if self.body is None:
            self.body = nn.Sequential(
                nn.Conv2d(cin, 32, (3, 7), 1, (1, 3), bias=False), Norm(32), nn.ReLU(inplace=True),
                Block(32, 48, 2), Block(48, 48, 1),
                Block(48, 96, 2), Block(96, 96, 1),
                Block(96, 160, 2), Block(160, 160, 1),
                Block(160, 256, 2), Block(256, 256, 1))
        self.fmlp = nn.Sequential(nn.Linear(nf, 128), nn.ReLU(inplace=True))
        self.head = nn.Sequential(nn.Linear(dim + 128, 256), nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(256, NC))
    def forward(self, x, f, gaps):
        h = self.body(x)
        h = torch.cat([F.adaptive_avg_pool2d(h, 1).flatten(1), F.adaptive_max_pool2d(h, 1).flatten(1)], 1)
        return self.head(torch.cat([h, self.fmlp(f)], 1))

class TokenNet(nn.Module):
    def __init__(self, nf, d=192):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(5, 32, (3, 7), 1, (1, 3), bias=False), Norm(32), nn.ReLU(inplace=True),
                                 Block(32, 64, (2, 2)), Block(64, 96, (2, 2)), Block(96, 128, (2, 2)))
        self.proj = nn.Linear(128 * 16, d)
        self.type_emb = nn.Parameter(torch.zeros(2, d)); self.pos_emb = nn.Parameter(torch.zeros(6, d))
        self.time_proj = nn.Linear(1, d); self.cls = nn.Parameter(torch.zeros(1, 1, d))
        nn.init.normal_(self.type_emb, std=0.02); nn.init.normal_(self.pos_emb, std=0.02); nn.init.normal_(self.cls, std=0.02)
        layer = nn.TransformerEncoderLayer(d, 4, 2 * d, 0.1, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, 3, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.fmlp = nn.Sequential(nn.Linear(nf, 128), nn.ReLU(inplace=True))
        self.head = nn.Sequential(nn.Linear(d + 128, 256), nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(256, NC))
    def forward(self, x, f, gaps):
        B = x.shape[0]
        h = self.enc(x.reshape(B * 6, 5, 16, 128))
        h = self.proj(h.mean(2).flatten(1)).view(B, 6, -1)
        p = torch.arange(6, device=x.device)
        start = (p.view(1, 6) * (16 + gaps.view(B, 1)).float() / 126.0).unsqueeze(-1)
        h = h + self.type_emb[(p % 2)].unsqueeze(0) + self.pos_emb.unsqueeze(0) + self.time_proj(start)
        h = self.tf(torch.cat([self.cls.expand(B, -1, -1), h], 1))
        return self.head(torch.cat([self.norm(h[:, 0]), self.fmlp(f)], 1))

def make_model(kind):
    return CanvasNet(NF) if kind == 'A' else TokenNet(NF)

def model_inputs(kind, prep, gaps):
    return canvas_view(prep, gaps) if kind == 'A' else token_view(prep)

log(f'model A: {sum(p.numel() for p in make_model("A").parameters()) / 1e6:.2f} M params')

In [ ]:
def class_weight_tensor(yv):
    cnt = np.bincount(yv, minlength=NC).astype(float)
    w = (cnt.sum() / (NC * np.maximum(cnt, 1))) ** 0.5
    return torch.tensor(w / w.mean(), dtype=torch.float32, device=DEVICE)

def forward_batch(model, kind, v, miss, g):
    prep = preprocess(v, miss)
    f = fnorm(featurize(prep, g)[0])
    x = model_inputs(kind, prep, g)
    with amp_ctx():
        return model(x, f, g).float()

TTA_VIEWS = [(False, 0), (True, 0), (False, 8), (False, -8)] if CONFIG['TTA'] else [(False, 0)]

@torch.no_grad()
def predict(model, kind, raw, gaps, idx, views=None, seed=0):
    model.eval(); views = views or TTA_VIEWS
    gen = torch.Generator(device=DEVICE).manual_seed(4242 + seed)
    out = np.zeros((len(idx), NC))
    for flip, sh in views:
        ps = []
        for st in range(0, len(idx), CONFIG['PRED_BS']):
            bi = torch.as_tensor(idx[st:st + CONFIG['PRED_BS']], device=DEVICE, dtype=torch.long)
            v = raw[bi].float(); miss = v == 0; B = len(bi)
            if flip or sh:
                v, miss = freq_resample(v, miss, torch.ones(B, device=DEVICE), torch.full((B,), float(sh), device=DEVICE),
                                        torch.full((B,), bool(flip), device=DEVICE), gen)
            ps.append(torch.softmax(forward_batch(model, kind, v, miss, gaps[bi]), 1).cpu().numpy())
        out += np.concatenate(ps)
    return out / len(views)

def has_bn(model):
    return any(isinstance(m, nn.modules.batchnorm._BatchNorm) for m in model.modules())

@torch.no_grad()
def adabn(model, kind, raw, gaps, idx):
    """AdaBN: recompute BatchNorm running statistics on target-domain inputs (cumulative average, no gradients)."""
    bns = [m for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)]
    for m in bns:
        m.reset_running_stats(); m.momentum = None
    model.eval()
    for m in bns:
        m.train()
    for st in range(0, len(idx), CONFIG['PRED_BS']):
        bi = torch.as_tensor(idx[st:st + CONFIG['PRED_BS']], device=DEVICE, dtype=torch.long)
        v = raw[bi].float()
        forward_batch(model, kind, v, v == 0, gaps[bi])
    model.eval()

def train_model(kind, seed, fold, raw, gaps, ys, ws, tr_idx, pool_idx, epochs, lr, init_state=None, va_idx=None):
    seed_everything(seed * 100 + fold)
    model = make_model(kind).to(DEVICE)
    if init_state is not None:
        model.load_state_dict(init_state)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CONFIG['WD'])
    BS = CONFIG['BS']; nb = max(1, len(tr_idx) // BS)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * nb, pct_start=0.15,
                                                div_factor=10.0, final_div_factor=100.0)
    scaler = make_scaler()
    gen = torch.Generator(device=DEVICE).manual_seed(seed * 1000 + fold)
    rng = np.random.RandomState(seed * 1000 + fold)
    cw = class_weight_tensor(ys[torch.as_tensor(tr_idx, device=DEVICE)].cpu().numpy())
    sd = model.state_dict()
    fkeys = [k for k, v in sd.items() if v.dtype.is_floating_point]
    ema = {k: v.detach().clone() for k, v in sd.items()} if CONFIG['EMA_DECAY'] else None
    if ema is not None:
        ema_f, src_f = [ema[k] for k in fkeys], [sd[k] for k in fkeys]
    step = 0
    for ep in range(epochs):
        model.train(); perm = rng.permutation(tr_idx); tl = 0.0; t_ep = time.time()
        for b in range(nb):
            bi = perm[b * BS:(b + 1) * BS]
            with torch.no_grad():
                v, miss, yb, wb, gb = make_train_batch(raw, gaps, ys, ws, bi, pool_idx, gen)
                prep = preprocess(v, miss)
                f = fnorm(featurize(prep, gb)[0])
                x = model_inputs(kind, prep, gb)
            with amp_ctx():
                logits = model(x, f, gb)
            loss = (F.cross_entropy(logits.float(), yb, weight=cw, label_smoothing=CONFIG['LABEL_SMOOTH'],
                                    reduction='none') * wb).mean()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt); scaler.update(); sched.step()
            step += 1
            if ema is not None:
                d = min(CONFIG['EMA_DECAY'], (1.0 + step) / (10.0 + step))
                with torch.no_grad():
                    torch._foreach_mul_(ema_f, d)
                    torch._foreach_add_(ema_f, src_f, alpha=1.0 - d)
            tl += float(loss)
        if va_idx is not None and ((ep + 1) % 5 == 0 or ep == epochs - 1):
            pv = predict(model, kind, TR_RAW, TR_GAPS, va_idx, views=[(False, 0)])
            log(f'  {kind} s{seed} f{fold} ep {ep + 1:2d}/{epochs} loss {tl / nb:.4f} clean-val F1 {mf1(y[va_idx], pv.argmax(1)):.4f} ({time.time() - t_ep:.0f}s/ep)')
        elif va_idx is None and ((ep + 1) % 10 == 0 or ep == epochs - 1):
            log(f'  {kind} s{seed} full-train ep {ep + 1:2d}/{epochs} loss {tl / nb:.4f} ({time.time() - t_ep:.0f}s/ep)')
    if ema is not None:
        for k, v in sd.items():
            if not v.dtype.is_floating_point:
                ema[k].copy_(v)
        if va_idx is not None:
            f_raw = mf1(y[va_idx], predict(model, kind, TR_RAW, TR_GAPS, va_idx, views=[(False, 0)]).argmax(1))
        model.load_state_dict(ema)
        if va_idx is not None:
            f_ema = mf1(y[va_idx], predict(model, kind, TR_RAW, TR_GAPS, va_idx, views=[(False, 0)]).argmax(1))
            log(f'  {kind} s{seed} f{fold}: clean-val F1 raw weights {f_raw:.4f} -> EMA weights {f_ema:.4f}')
    return model

def fold_outputs(model, kind, va, te_idx):
    """clean / PKS-sim / PKS-sim-R OOF and test probabilities, plain and AdaBN-adapted (TTA everywhere)."""
    o = dict(clean=predict(model, kind, TR_RAW, TR_GAPS, va), pks=predict(model, kind, PKS_RAW, TR_GAPS, va),
             pksR=predict(model, kind, PKSR_RAW, TR_GAPS, va), test=predict(model, kind, TE_RAW, TE_GAPS, te_idx))
    if has_bn(model):
        state = copy.deepcopy(model.state_dict())
        for key, R, G, ix in (('pks_ada', PKS_RAW, TR_GAPS, va), ('pksR_ada', PKSR_RAW, TR_GAPS, va),
                              ('test_ada', TE_RAW, TE_GAPS, te_idx)):
            adabn(model, kind, R, G, ix)
            o[key] = predict(model, kind, R, G, ix)
            model.load_state_dict(state)
    else:
        o.update(pks_ada=o['pks'], pksR_ada=o['pksR'], test_ada=o['test'])
    return o

VARIANTS = ('clean', 'pks', 'pksR', 'test', 'pks_ada', 'pksR_ada', 'test_ada')
TE_IDX = np.arange(N_TE)
POOL_ALL = np.where(np.isin(y, PARTNER_CLASSES))[0]

def run_cv(kind, seed, epochs, lr):
    acc = {k: np.zeros((N_TR, NC)) if k not in ('test', 'test_ada') else np.zeros((N_TE, NC)) for k in VARIANTS}
    t_fold = []
    for f in range(CONFIG['FOLDS']):
        t0 = time.time()
        tr, va = np.where(FOLD != f)[0], np.where(FOLD == f)[0]
        pool = POOL_ALL[FOLD[POOL_ALL] != f]
        model = train_model(kind, seed, f, TR_RAW, TR_GAPS, TR_Y, TR_W, tr, pool, epochs, lr, va_idx=va)
        o = fold_outputs(model, kind, va, TE_IDX)
        for k in VARIANTS:
            if k.startswith('test'):
                acc[k] += o[k] / CONFIG['FOLDS']
            else:
                acc[k][va] = o[k][: len(va)]
        del model; gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
        t_fold.append(time.time() - t0)
        log(f'{kind} seed {seed} fold {f}: clean {mf1(y[va], acc["clean"][va].argmax(1)):.4f} | PKS-sim {mf1(y[va], acc["pks"][va].argmax(1)):.4f} '
            f'| PKS-sim+AdaBN {mf1(y[va], acc["pks_ada"][va].argmax(1)):.4f} | {t_fold[-1] / 60:.1f} min')
    return acc, float(np.mean(t_fold))

def pick_ada(name, acc):
    """Choose plain vs AdaBN by PKS-sim OOF F1 and return the dict used downstream."""
    f_plain, f_ada = mf1(y, acc['pks'].argmax(1)), mf1(y, acc['pks_ada'].argmax(1))
    use = f_ada > f_plain
    log(f'{name}: PKS-sim plain {f_plain:.4f} vs AdaBN {f_ada:.4f} -> {"AdaBN" if use else "plain"}')
    sfx = '_ada' if use else ''
    return dict(clean=acc['clean'], pks=acc['pks' + sfx], pksR=acc['pksR' + sfx], test=acc['test' + sfx], ada=use, raw=acc)

def minutes_left():
    return CONFIG['TIME_BUDGET_MIN'] - CONFIG['SAFETY_MIN'] - elapsed_min()

def full_model_test(seed, tag, raw=None, gaps=None, ys=None, ws=None, tr_idx=None):
    """Train Model A on all rows of `raw` (train, optionally + pseudo-labelled test) and return TTA test probabilities."""
    raw = TR_RAW if raw is None else raw; gaps = TR_GAPS if gaps is None else gaps
    ys = TR_Y if ys is None else ys; ws = TR_W if ws is None else ws
    tr_idx = np.arange(N_TR) if tr_idx is None else tr_idx
    t0 = time.time()
    model = train_model('A', seed, 99, raw, gaps, ys, ws, tr_idx, POOL_ALL, CONFIG['EPOCHS_A'], CONFIG['LR_A'])
    if RESULTS['A']['ada'] and has_bn(model):
        adabn(model, 'A', TE_RAW, TE_GAPS, TE_IDX)
    p = predict(model, 'A', TE_RAW, TE_GAPS, TE_IDX)
    torch.save(model.state_dict(), os.path.join(WORK, f'modelA_{tag}.pt'))
    del model; gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    m = (time.time() - t0) / 60
    log(f'{tag}: trained on {len(tr_idx)} rows in {m:.1f} min')
    return p, m

In [ ]:
# ===================== 5a. Model A, 5-fold CV (the OOF every decision is fitted on) =====================
t_a = time.time()
accA, tfoldA = run_cv('A', CONFIG['SEEDS_A'][0], CONFIG['EPOCHS_A'], CONFIG['LR_A'])
RUNTIME['A'] = (time.time() - t_a) / 60
RESULTS['A'] = pick_ada('A', accA)
SUMMARY_ROWS.insert(0, three_numbers('A (canvas)', RESULTS['A']))
print_f1('Model A clean OOF', y, RESULTS['A']['clean'].argmax(1))
print_f1('Model A PKS-sim OOF', y, RESULTS['A']['pks'].argmax(1))
np.savez_compressed(os.path.join(WORK, 'v2_modelA_cv.npz'), **{k: v.astype(np.float32) for k, v in accA.items()})
STAGE_TIMES['model_A'] = elapsed_min()
log(f'Model A CV done in {RUNTIME["A"]:.1f} min ({tfoldA / 60:.1f} min/fold); {minutes_left():.0f} min left in budget')

In [ ]:
# ===================== 5b. optional Model B (panel tokens) =====================
if CONFIG['USE_MODEL_B']:
    t_b = time.time()
    accB, _ = run_cv('B', CONFIG['SEEDS_A'][0], CONFIG['EPOCHS_B'], CONFIG['LR_B'])
    RUNTIME['B'] = (time.time() - t_b) / 60
    RESULTS['B'] = pick_ada('B', accB)
    SUMMARY_ROWS.append(three_numbers('B (tokens)', RESULTS['B']))
    STAGE_TIMES['model_B'] = elapsed_min()
else:
    log('Model B skipped (USE_MODEL_B=False)')

## 6. Decision layer (fitted once on the CV OOF, PKS-sim is the decision number)
Temperatures and blend weights (A + C [+ B]) are fitted on the PKS-sim OOF, then the per-class logit biases (kept only if a cross-fitted check gains), the Saerens EM prior correction (kept only if the stress test improves) and the eti physics override. The same decision layer is then applied to every later test ensemble.

In [ ]:
def temper(p, T):
    lp = np.log(np.clip(p, 1e-9, 1)) / T
    lp -= lp.max(1, keepdims=True); e = np.exp(lp)
    return e / e.sum(1, keepdims=True)

def fit_T(p, yt):
    best_T, best = 1.0, 1e9
    for T in np.arange(0.5, 3.01, 0.1):
        nll = -np.mean(np.log(np.clip(temper(p, T)[np.arange(len(yt)), yt], 1e-9, 1)))
        if nll < best:
            best, best_T = nll, float(T)
    return best_T

def simplex_grid(n, step):
    k = int(round(1 / step))
    if n == 1:
        return [np.array([1.0])]
    out = []
    for comb in itertools.product(range(k + 1), repeat=n - 1):
        if sum(comb) <= k:
            out.append(np.array(list(comb) + [k - sum(comb)], float) / k)
    return out

def blend(names):
    Ts = {n: fit_T(RESULTS[n]['pks'], y) for n in names}
    tp = {n: {k: temper(RESULTS[n][k], Ts[n]) for k in ('clean', 'pks', 'pksR', 'test')} for n in names}
    best_w, best = None, -1
    for w in simplex_grid(len(names), 0.05 if len(names) <= 3 else 0.1):
        s = mf1(y, sum(wi * tp[n]['pks'] for wi, n in zip(w, names)).argmax(1))
        if s > best + 1e-9:
            best, best_w = s, w
    ens = {k: sum(wi * tp[n][k] for wi, n in zip(best_w, names)) for k in ('clean', 'pks', 'pksR', 'test')}
    ens.update(names=names, weights=best_w, T=Ts)
    log(f'blend {names}: T={ {n: round(t, 2) for n, t in Ts.items()} } weights={np.round(best_w, 2).tolist()} PKS-sim F1 {best:.4f}')
    return ens

def fit_bias(P, yt, rounds=3, grid=np.round(np.arange(-1.0, 1.0001, 0.05), 2)):
    lp = np.log(np.clip(P, 1e-9, 1)); b = np.zeros(NC); best = mf1(yt, lp.argmax(1))
    for _ in range(rounds):
        for c in range(NC):
            for val in grid:
                b2 = b.copy(); b2[c] = val
                s = mf1(yt, (lp + b2).argmax(1))
                if s > best + 1e-6:
                    best, b = s, b2
    return b

def crossfit_bias_gain(P):
    skf = StratifiedKFold(2, shuffle=True, random_state=SEED); gains = []
    for a, bix in skf.split(P, y):
        b = fit_bias(P[a], y[a])
        lp = np.log(np.clip(P[bix], 1e-9, 1))
        gains.append(mf1(y[bix], (lp + b).argmax(1)) - mf1(y[bix], lp.argmax(1)))
    return float(np.mean(gains))

IDX_ONMIN, IDX_OFFMAX, IDX_ABSD = [FEAT_NAMES.index(n) for n in ('gt_con_onmin', 'gt_con_offmax', 'gt_con_absd')]

def apply_override(pred, scores, Fm):
    cond = (pred == 0) & ((Fm[:, IDX_ONMIN] < 3) | (Fm[:, IDX_OFFMAX] > 4) | (Fm[:, IDX_ABSD] < 0.1))
    second = np.argsort(scores, 1)[:, -2]
    out = pred.copy(); out[cond] = second[cond]
    return out, int(cond.sum())

def finalize(ens, tag):
    P = ens['pks']; src = P.mean(0)
    zero = np.zeros(NC)
    bias = fit_bias(P, y)
    gain = crossfit_bias_gain(P)
    s_plain = stress_f1(P, zero)[0]; s_bias = stress_f1(P, bias)[0]
    use_bias = (gain > 0.001) and (s_bias >= s_plain - 0.003)
    b = bias if use_bias else zero
    s_noem, v_noem = stress_f1(P, b, False, src)
    s_em, v_em = stress_f1(P, b, True, src)
    use_em = s_em > s_noem + 0.002
    pred_pks, sc_pks = decide(P, b, use_em, src)
    pred_clean, sc_clean = decide(ens['clean'], b, False, src)
    use_ov, n_ov = False, 0
    if CONFIG['PHYSICS_OVERRIDE']:
        ov, n_ov = apply_override(pred_pks, sc_pks, F_pks)
        use_ov = mf1(y, ov) > mf1(y, pred_pks) + 0.0005
        if use_ov:
            pred_pks = ov
            pred_clean = apply_override(pred_clean, sc_clean, F_tr)[0]
    predR, _ = decide(ens['pksR'], b, use_em, src)
    out = dict(tag=tag, use_bias=use_bias, bias=b, use_em=use_em, use_override=use_ov, src_prior=src,
               clean=mf1(y, pred_clean), pks_sim=mf1(y, pred_pks), pks_sim_R=mf1(y, predR),
               stress=s_em if use_em else s_noem, stress_parts=v_em if use_em else v_noem, pred_pks=pred_pks)
    log(f'[{tag}] bias: crossfit gain {gain:+.4f}, stress {s_plain:.4f}->{s_bias:.4f} -> {"USED" if use_bias else "not used"} '
        f'| EM: stress {s_noem:.4f}->{s_em:.4f} -> {"USED" if use_em else "not used"} '
        f'| override ({n_ov} eti flips on PKS-sim) -> {"USED" if use_ov else "not used"}')
    log(f'[{tag}] OOF clean {out["clean"]:.4f} | PKS-sim {out["pks_sim"]:.4f} (LB proxy) | PKS-sim-R {out["pks_sim_R"]:.4f} '
        f'| stress {out["stress"]:.4f}')
    return out

NAMES = ['A', 'C'] + (['B'] if 'B' in RESULTS else [])
ENS0 = blend(NAMES)
CFG0 = finalize(ENS0, 'cv-oof')
print_f1('ensemble PKS-sim OOF', y, CFG0['pred_pks'])
print_cm('PKS-sim OOF, ensemble', y, CFG0['pred_pks'])
W0 = dict(zip(ENS0['names'], ENS0['weights']))
STAGE_TIMES['blend'] = elapsed_min()

## 7. Full-train seeds + noisy student on the real PKS test
Each extra seed trains Model A on **all** 18,000 training cadences. Each noisy-student seed also trains on the confident (> 0.90, at most 1,500 per class) **pseudo-labelled test cadences** of the current best ensemble (weight 0.7), under the same strong augmentation. This is the only stage that learns from real PKS noise, bandpass and label mix. The PKS-sim OOF cannot measure that, so the leaderboard decides: `submission.csv` includes the noisy-student models, and `submission_alt.csv` leaves them out.
The test ensemble = mean of the Model A test probabilities (5-fold CV average + full-train + noisy-student models). It is blended with C through the OOF-fitted temperatures and weights, then goes through the same decision layer. Files are rewritten after **every** model.

In [ ]:
def write_sub(pred, fname):
    sub = pd.DataFrame({'id': test_df['id'].values, 'label': [CLASSES[i] for i in pred]})
    ref = pd.read_csv(os.path.join(DATA, 'test_meta.csv'))['id'].values
    assert list(sub.columns) == ['id', 'label']
    assert len(sub) == N_TE and sub['id'].is_unique
    if not CONFIG['FAST_DEBUG']:
        assert len(sub) == 10000, len(sub)
        assert (sub['id'].values == ref).all(), 'ids are not in test_meta.csv order'
    else:
        assert (sub['id'].values == ref[:N_TE]).all()
    assert sub['label'].isin(CLASSES).all()
    path = os.path.join(WORK, fname if not CONFIG['FAST_DEBUG'] else fname.replace('.csv', '_debug.csv'))
    sub.to_csv(path, index=False)
    return path

def ens_test(A_test):
    p = W0['A'] * temper(A_test, ENS0['T']['A'])
    for n in ENS0['names']:
        if n != 'A':
            p = p + W0[n] * temper(RESULTS[n]['test'], ENS0['T'][n])
    return p

def decide_test(p):
    pred, sc = decide(p, CFG0['bias'], CFG0['use_em'], CFG0['src_prior'])
    if CFG0['use_override']:
        pred = apply_override(pred, sc, F_te)[0]
    return pred

A_TESTS = {'cv5': RESULTS['A']['test']}     # model name -> Model A test probabilities
FAMILY = {'cv5': 'base'}                    # 'base' (train only) or 'pl' (noisy student)
LAST = {}

def emit():
    base = [A_TESTS[k] for k in A_TESTS if FAMILY[k] == 'base']
    out = {'nopseudo': ens_test(np.mean(base, 0)), 'pseudo': ens_test(np.mean(list(A_TESTS.values()), 0))}
    preds = {k: decide_test(v) for k, v in out.items()}
    has_pl = any(f == 'pl' for f in FAMILY.values())
    write_sub(preds['nopseudo'], 'submission_v2_nopseudo.csv')
    write_sub(preds['pseudo'], 'submission_v2_pseudo.csv')
    write_sub(preds['pseudo'] if has_pl else preds['nopseudo'], 'submission.csv')
    write_sub(preds['nopseudo'], 'submission_alt.csv')
    cv_pred = decide_test(ens_test(A_TESTS['cv5']))
    log(f'emit: models {list(A_TESTS)} | pseudo vs no-pseudo labels differ {100 * (preds["pseudo"] != preds["nopseudo"]).mean():.2f}% '
        f'| vs CV-only {100 * (preds["nopseudo"] != cv_pred).mean():.2f}%')
    np.savez_compressed(os.path.join(WORK, 'oof_and_test_probs.npz'),
                        y=y, fold=FOLD, classes=np.array(CLASSES), test_ids=test_df['id'].values.astype(str),
                        feat_names=np.array(FEAT_NAMES), F_tr=F_tr.astype(np.float32), F_pks=F_pks.astype(np.float32),
                        F_te=F_te.astype(np.float32), bias=CFG0['bias'], src_prior=CFG0['src_prior'],
                        ens_weights=ENS0['weights'], ens_names=np.array(ENS0['names']),
                        **{f'A_test_{k}': v.astype(np.float32) for k, v in A_TESTS.items()},
                        **{f'{n}_{k}': RESULTS[n][k].astype(np.float32) for n in RESULTS for k in ('clean', 'pks', 'pksR', 'test')},
                        ens_test_nopseudo=out['nopseudo'].astype(np.float32), ens_test_pseudo=out['pseudo'].astype(np.float32))
    LAST.update(out=out, preds=preds, has_pl=has_pl)
    return out, preds

emit()
log('wrote submission files from the CV models (safe fallback)')

# per-model agreement check (a broken model shows up as low agreement with the CV ensemble)
def agree(p):
    return float((p.argmax(1) == A_TESTS['cv5'].argmax(1)).mean())

PL_LOG = []
est_full = tfoldA / 60 * 1.25 + 1.0          # a full-train model ~ one CV fold on 1.25x the data
for s in CONFIG['FULL_SEEDS']:
    # ---- full-train seed (train only) ----
    if est_full > minutes_left():
        log(f'time guard: stopping before full-train seed {s} (needs ~{est_full:.0f} min, {minutes_left():.0f} left)'); break
    p, m = full_model_test(s, f'full_s{s}')
    A_TESTS[f'full_s{s}'] = p; FAMILY[f'full_s{s}'] = 'base'; RUNTIME[f'full_s{s}'] = m
    log(f'full_s{s}: test argmax agreement with CV ensemble {agree(p):.3f}')
    emit()
    # ---- noisy-student seed (train + pseudo-labelled real test) ----
    if not CONFIG['USE_PSEUDO']:
        continue
    est_pl = est_full * 1.4
    if est_pl > minutes_left():
        log(f'time guard: stopping before noisy-student seed {s} (needs ~{est_pl:.0f} min, {minutes_left():.0f} left)'); break
    teacher = LAST['out']['pseudo']
    q = em_prior(teacher, CFG0['src_prior'])[0] if CFG0['use_em'] else teacher
    conf, lab = q.max(1), q.argmax(1)
    pl, counts = [], {}
    for c in range(NC):
        ic = np.where((lab == c) & (conf > CONFIG['PSEUDO_THR']))[0]
        ic = ic[np.argsort(-conf[ic])][:CONFIG['PSEUDO_CAP']]
        pl.append(ic); counts[CLASSES[c]] = len(ic)
    PL_IDX = np.sort(np.concatenate(pl))
    log(f'noisy student s{s}: {len(PL_IDX)} pseudo-labelled test cadences {counts}')
    PL_LOG.append(dict(seed=s, n=len(PL_IDX), **counts))
    if len(PL_IDX) == 0:
        continue
    pl_t = torch.as_tensor(PL_IDX, device=DEVICE, dtype=torch.long)
    COMB_RAW = torch.cat([TR_RAW, TE_RAW[pl_t]])
    COMB_GAPS = torch.cat([TR_GAPS, TE_GAPS[pl_t]])
    COMB_Y = torch.cat([TR_Y, torch.as_tensor(lab[PL_IDX], device=DEVICE, dtype=torch.long)])
    COMB_W = torch.cat([TR_W, torch.full((len(PL_IDX),), CONFIG['PSEUDO_W'], device=DEVICE)])
    p, m = full_model_test(s + 7, f'pl_s{s}', raw=COMB_RAW, gaps=COMB_GAPS, ys=COMB_Y, ws=COMB_W,
                           tr_idx=np.arange(N_TR + len(PL_IDX)))
    del COMB_RAW, COMB_GAPS, COMB_Y, COMB_W; gc.collect()
    A_TESTS[f'pl_s{s}'] = p; FAMILY[f'pl_s{s}'] = 'pl'; RUNTIME[f'pl_s{s}'] = m
    log(f'pl_s{s}: test argmax agreement with CV ensemble {agree(p):.3f}')
    emit()
STAGE_TIMES['full_and_pseudo'] = elapsed_min()

## 8. Summary
The OOF numbers (clean / PKS-sim / stress) are only defined for the CV models. The full-train and noisy-student models add test-only probabilities, and the leaderboard is their validation. v1 reference: PKS-sim 0.9016 → public LB 0.9029.

In [ ]:
rows = [dict(r, runtime_min=round(RUNTIME.get(r['model'].split(' ')[0], float('nan')), 1)) for r in SUMMARY_ROWS]
rows.append(dict(model='ENSEMBLE (OOF decision layer)', clean=CFG0['clean'], pks_sim=CFG0['pks_sim'],
                 pks_sim_R=CFG0['pks_sim_R'], stress=CFG0['stress'], runtime_min=round(elapsed_min(), 1)))
print('=' * 100)
print('SUMMARY  (decision number = pks_sim; v1: PKS-sim 0.9016 -> public LB 0.9029)')
print(pd.DataFrame(rows).set_index('model').round(4).to_string())
print('\ntest models in the ensemble:', {k: FAMILY[k] for k in A_TESTS})
print('model runtimes (min):', {k: round(v, 1) for k, v in RUNTIME.items()})
if PL_LOG:
    print('pseudo-label rounds:'); print(pd.DataFrame(PL_LOG).to_string(index=False))
print('\npredicted test label distribution:')
print(pd.DataFrame({'train_frac': np.bincount(y, minlength=NC) / N_TR,
                    'v2_pseudo (submission.csv)': np.bincount(LAST['preds']['pseudo'], minlength=NC) / N_TE,
                    'v2_nopseudo (submission_alt.csv)': np.bincount(LAST['preds']['nopseudo'], minlength=NC) / N_TE,
                    'EM test prior': em_prior(LAST['out']['pseudo'], CFG0['src_prior'])[1]}, index=CLASSES).round(3).to_string())
print(f'\nlabels that differ between submission.csv and submission_alt.csv: '
      f'{100 * float((LAST["preds"]["pseudo"] != LAST["preds"]["nopseudo"]).mean()):.2f}%')
print('decision layer:', dict(bias=CFG0['use_bias'], em=CFG0['use_em'], override=CFG0['use_override'],
                              ensemble=ENS0['names'], weights=np.round(ENS0['weights'], 2).tolist(),
                              adabn_A=RESULTS['A']['ada']))
print('stage end times (min):', {k: round(v, 1) for k, v in STAGE_TIMES.items()})
json.dump(dict(cfg=dict(bias=bool(CFG0['use_bias']), em=bool(CFG0['use_em']), override=bool(CFG0['use_override'])),
               oof=dict(clean=CFG0['clean'], pks_sim=CFG0['pks_sim'], stress=CFG0['stress']),
               models={k: FAMILY[k] for k in A_TESTS}, pseudo=PL_LOG, runtime_min=elapsed_min(), config=CONFIG),
          open(os.path.join(WORK, 'run_summary.json'), 'w'), indent=1, default=str)
log('done')